# M5.7 · GAN II — DCGAN, FID/IS, progresif büyütme, StyleGAN

GAN'ları pratikte eğitmek ve değerlendirmek: Nash dengesi, mod çöküşü ve çevrimler, deneyim tekrarı ve mini-batch ayrımı, doymayan üretici kaybı; DCGAN kuralları (havuz yerine adımlı/transposed konvolüsyon, BN, ReLU/tanh, LeakyReLU); ne zaman durmalı — Inception Score ve Fréchet Inception Distance (Gauss varsayımı, matris karekökü); ProGAN'ın progresif büyütmesi (yumuşak geçiş), mini-batch standart sapma katmanı, piksel normalizasyonu ve eşitlenmiş öğrenme oranı; StyleGAN'ın eşleme ağı, AdaIN, gürültü enjeksiyonu, stil karıştırma ve kırpma (truncation) hilesi. NumPy'da FID'i sıfırdan, PyTorch'ta rakamlar üzerinde DCGAN ve sınıflandırıcı öznitelikleriyle FID/kapsama, sonda StyleGAN blokları ve StyleGAN2'nin demodülasyonu. Ekler: IS'in kandırılması, FID'in örneklem yanlılığı, ProGAN hileleri.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-7-gan-advanced.html](https://m-gonul.github.io/pytorch-lab/m5-7-gan-advanced.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_7_gan_advanced.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        close(need(ns, \"frechet_1d\")(0, 1, 1, 2), 2.0, what=\"FID 1B\")\n        close(ns[\"frechet_1d\"](3, 2, 3, 2), 0.0, what=\"aynı dağılım\")\n        close(need(ns, \"inception_score\")([[0.9, 0.1], [0.1, 0.9]]), math.exp(0.9 * math.log(1.8) + 0.1 * math.log(0.2)), what=\"IS\")\n        close(ns[\"inception_score\"]([[1, 0, 0], [0, 1, 0], [0, 0, 1]]), 3.0, what=\"mükemmel ve dengeli: sınıf sayısı\")\n        close(ns[\"inception_score\"]([[1, 0, 0]] * 5), 1.0, what=\"tek sınıf: 1\")\n        y = need(ns, \"adain\")([1, 2, 3, 4], 2, 1)\n        close(y[3], 2 * 1.5 / math.sqrt(1.25) + 1, what=\"AdaIN\")\n        m = sum(y) / 4\n        close((m, math.sqrt(sum((v - m) ** 2 for v in y) / 4)), (1.0, 2.0), what=\"çıktı ortalama y_b, std y_s\")\n        close(need(ns, \"truncate\")([2.0, -1.0], [0.0, 1.0], 0.7), [1.4, -0.4], what=\"kırpma\")\n        close(need(ns, \"fade_in\")([2.0], [4.0], 0.3), [2.6], what=\"yumuşak geçiş\")\n        close(need(ns, \"he_scale\")(512), math.sqrt(2 / 512), what=\"He ölçeği\")\n        rows = []\n        for mu, s in ((0, 1), (0.5, 1), (0, 0.5), (0, 2), (2, 1)):\n            rows.append([mu, s, f\"{ns['frechet_1d'](0, 1, mu, s):.2f}\"])\n        R_.table(\"Gerçek N(0, 1) ile üretilen N(μ, σ²) arasındaki Fréchet uzaklığı\", [\"μ\", \"σ\", \"FID (1B)\"], rows,\n                 note=\"FID ortalama kaymasını ve yayılım farkını birlikte cezalandırır: σ = 0.5 (çeşitlilik kaybı, mod çöküşünün 1B karşılığı) 0.25, σ = 2 (aşırı yayılma) 1.0 ceza alır.\")\n    R_.test(\"formüller\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüler\", lambda: (max_loops(src, 0, func=\"sqrtm_psd\"), max_loops(src, 0, func=\"frechet\"), max_loops(src, 1, func=\"sqrtm_ns\")))\n    rng = np.random.default_rng(0)\n\n    def t():\n        A = rng.normal(size=(6, 6))\n        S = A @ A.T + 0.1 * np.eye(6)\n        R = need(ns, \"sqrtm_psd\")(S)\n        close(R @ R, S, atol=1e-9, what=\"√S · √S = S\")\n        close(need(ns, \"sqrtm_ns\")(S, 60), R, atol=1e-6, what=\"Newton–Schulz\")\n        close(need(ns, \"frechet\")(np.array([1.0, 0]), np.eye(2), np.zeros(2), np.diag([4.0, 9])), 6.0, what=\"el örneği\")\n        S2 = (lambda B: B @ B.T)(rng.normal(size=(6, 6)))\n        w = np.linalg.eigvals(S @ S2)\n        close(ns[\"frechet\"](np.zeros(6), S, np.zeros(6), S2), np.trace(S) + np.trace(S2) - 2 * np.sum(np.sqrt(w.real.clip(0))), atol=1e-6, what=\"tr√(S₁S₂) simetrik biçimle aynı\")\n        P = np.eye(4)[rng.integers(0, 4, 4000)]\n        close(need(ns, \"inception_score\")(P, 1)[0], 4.0, atol=0.02, what=\"dengeli ve emin → C\")\n        centers = rng.normal(0, 3, (10, 16))\n        real = centers[rng.integers(0, 10, 5000)] + rng.normal(size=(5000, 16))\n        rows = []\n        for name, fake in ((\"gerçekle aynı dağılım\", centers[rng.integers(0, 10, 5000)] + rng.normal(size=(5000, 16))),\n                           (\"sınıf içi std 0.5\", centers[rng.integers(0, 10, 5000)] + 0.5 * rng.normal(size=(5000, 16))),\n                           (\"5 mod (yarısı düştü)\", centers[rng.integers(0, 5, 5000)] + rng.normal(size=(5000, 16))),\n                           (\"1 mod\", centers[np.zeros(5000, int)] + rng.normal(size=(5000, 16))),\n                           (\"ortalama 0.5 kaymış\", centers[rng.integers(0, 10, 5000)] + 0.5 + rng.normal(size=(5000, 16)))):\n            rows.append([name, f\"{need(ns, 'fid')(real, fake):.2f}\"])\n        f = [float(r[1]) for r in rows]\n        ensure(f[0] < 0.2 and f[0] < f[4] < f[3] and f[2] < f[3] and f[1] > f[0], f\"FID sırası {f}\")\n        R_.table(\"16 boyutlu öznitelik uzayında 10 modlu 'gerçek' veri\", [\"üretilen\", \"FID\"], rows,\n                 note=\"FID tek bir Gauss uydurur: mod düşürmek ortalamayı ve kovaryansı değiştirdiği için yakalanır, ama çok modluluğun kendisini değil yalnız ilk iki momentini görür. Aynı dağılımdan çekilen iki küme bile > 0 verir (sonlu örneklem yanlılığı, x2).\")\n    R_.test(\"matris karekökü ve FID\", t)\n",
 "d": "import time\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    X, y = need(ns, \"digits16\")()\n    torch.manual_seed(0)\n    clf = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n                        nn.Flatten(), nn.Linear(1024, 64), nn.ReLU(), nn.Linear(64, 10))\n    o = torch.optim.Adam(clf.parameters(), 1e-3)\n    for _ in range(15):\n        for i in range(0, len(X), 64):\n            p = torch.randperm(len(X))[:64]\n            loss = F.cross_entropy(clf(X[p]), y[p])\n            o.zero_grad()\n            loss.backward()\n            o.step()\n    clf.eval()\n\n    def feat(x):\n        with torch.no_grad():\n            return clf[:-1](x).numpy()\n\n    def t_arch():\n        G, D = need(ns, \"Generator\")(32, 64), need(ns, \"Discriminator\")(64)\n        x = G(torch.randn(5, 32))\n        ensure(tuple(x.shape) == (5, 1, 16, 16) and float(x.abs().max()) <= 1, \"üretici çıkışı (B, 1, 16, 16), tanh\")\n        ensure(tuple(D(x).shape) == (5,), \"ayırıcı logit\")\n        ensure(not any(isinstance(m, nn.MaxPool2d) for m in list(G.modules()) + list(D.modules())), \"havuz yok (DCGAN)\")\n        ensure(not isinstance(list(D.net)[1], nn.BatchNorm2d), \"ayırıcı girişinde BN yok\")\n    R_.test(\"DCGAN mimarisi\", t_arch)\n\n    def t_train():\n        fr = feat(X)\n        torch.manual_seed(0)\n        G, D = ns[\"Generator\"](32, 64), ns[\"Discriminator\"](64)\n        t0 = time.perf_counter()\n        G0 = ns[\"Generator\"](32, 64)\n        need(ns, \"train_gan\")(G, D, X, 30)\n        G.eval()\n        G0.eval()\n        with torch.no_grad():\n            xs, x0 = G(torch.randn(1000, 32)), G0(torch.randn(1000, 32))\n        rows = []\n        fd = need(ns, \"fid_np\")\n        sets = [(\"gerçek (yarı) vs gerçek (yarı)\", feat(X[::2]), None), (\"DCGAN, 30 epoch\", feat(xs), xs), (\"eğitilmemiş üretici\", feat(x0), x0),\n                (\"gerçek, yalnız 0–2 rakamları\", feat(X[y <= 2]), X[y <= 2]), (\"gerçek + N(0, 0.5²) gürültü\", feat((X + 0.5 * torch.randn(X.shape)).clamp(-1, 1)), None)]\n        res = {}\n        for name, f, imgs in sets:\n            ref = feat(X[1::2]) if name.startswith(\"gerçek (yarı)\") else fr\n            res[name] = fd(ref, f)\n            cov = \"—\"\n            if imgs is not None:\n                with torch.no_grad():\n                    c = torch.bincount(clf(imgs).argmax(1), minlength=10)\n                cov = f\"{int((c > 0.02 * len(imgs)).sum())}/10\"\n            rows.append([name, f\"{res[name]:.1f}\", cov])\n        ensure(res[\"DCGAN, 30 epoch\"] < 0.5 * res[\"eğitilmemiş üretici\"] and res[\"DCGAN, 30 epoch\"] > res[\"gerçek (yarı) vs gerçek (yarı)\"], f\"FID {res}\")\n        ensure(res[\"gerçek, yalnız 0–2 rakamları\"] > res[\"gerçek (yarı) vs gerçek (yarı)\"] * 3, \"mod düşürme FID'i büyütür\")\n        R_.table(f\"16 × 16 rakamlar: CNN sınıflandırıcı öznitelikleriyle FID ({time.perf_counter() - t0:.0f} sn eğitim)\", [\"örnekler\", \"FID\", \"sınıf kapsamı\"], rows,\n                 note=\"Gerçek veriyi ikiye bölmek bile sıfırdan uzak bir FID verir (sonlu örneklem yanlılığı + 900 örnek) — bu, ulaşılabilecek alt sınırdır. Eğitilmiş DCGAN tüm sınıfları kapsar ve FID'i rastgele üreticinin çok altına indirir; gerçek ama yalnız üç sınıflık görüntüler (mükemmel kalite, düşük çeşitlilik) FID'de kötüdür: FID çeşitliliği de ölçer.\")\n    R_.test(\"eğitim ve FID\", t_train)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        M = need(ns, \"MappingNet\")(16, 32, 8)\n        z = torch.randn(4, 16)\n        close(M(z).detach().numpy(), M(3 * z).detach().numpy(), atol=1e-5, what=\"piksel normu: z ölçeğinden bağımsız\")\n        ensure(sum(isinstance(m, torch.nn.Linear) for m in M.modules()) == 8, \"8 doğrusal katman\")\n        A = need(ns, \"AdaIN\")(6, 32)\n        x = torch.randn(4, 6, 8, 8) * 5 + 3\n        w = torch.randn(4, 32)\n        y = A(x, w)\n        ys, yb = A.style(w).chunk(2, 1)\n        close(y.mean((2, 3)).detach().numpy(), yb.detach().numpy(), atol=1e-4, what=\"çıktı ortalaması = y_b\")\n        close(y.std((2, 3), unbiased=False).detach().numpy(), ys.abs().detach().numpy(), atol=1e-3, what=\"çıktı std = |y_s|\")\n        close(A(x * 7 - 2, w).detach().numpy(), y.detach().numpy(), atol=1e-4, what=\"girdinin önceki stili silinir\")\n        Nz = need(ns, \"NoiseInjection\")(6)\n        ensure(torch.equal(Nz(x), x), \"ölçek 0 başlar → başlangıçta etkisiz\")\n        with torch.no_grad():\n            Nz.scale.fill_(0.5)\n        n = torch.randn(4, 1, 8, 8)\n        close(Nz(x, n).detach().numpy(), (x + 0.5 * n).numpy(), atol=1e-6, what=\"kanal başına ölçekli gürültü (kanallar arası aynı harita)\")\n        wt = torch.randn(5, 6, 3, 3)\n        s = torch.rand(4, 6) + 0.5\n        out = need(ns, \"modulated_conv\")(x, wt, s, False)\n        ref = torch.stack([F.conv2d((x[i:i + 1] * s[i][None, :, None, None]), wt, padding=1)[0] for i in range(4)])\n        close(out.numpy(), ref.numpy(), atol=1e-4, what=\"modülasyon = girdiyi kanal başına ölçeklemek\")\n        xs = torch.randn(4, 6, 16, 16)\n        od = ns[\"modulated_conv\"](xs, wt, s * 10, True)\n        close(od.std((0, 2, 3)).numpy(), torch.ones(5).numpy(), atol=0.15, what=\"demodülasyon: i.i.d. birim girdide çıktı std ≈ 1, stil ölçeğinden bağımsız\")\n        W = M(torch.randn(2000, 16)).detach()\n        wavg = W.mean(0)\n        rows = []\n        for psi in (1.0, 0.7, 0.5, 0.0):\n            Wt = need(ns, \"truncate\")(W, wavg, psi)\n            rows.append([psi, f\"{Wt.std(0).mean().item():.4f}\", f\"{(Wt - wavg).norm(dim=1).mean().item():.4f}\"])\n        ensure(float(rows[-1][1]) < 1e-6, \"ψ = 0: tek görüntü (ortalama)\")\n        R_.table(\"Kırpma hilesi: w uzayında çeşitlilik\", [\"ψ\", \"boyut başına std\", \"ort. ‖w − w̄‖\"], rows,\n                 note=\"Kırpma w'leri ortalamaya çeker: düşük olasılıklı (kötü üretilen) bölgelerden kaçınır, kalite artar, çeşitlilik ψ ile doğrusal azalır. StyleGAN2'de AdaIN'in istatistik normalizasyonu ağırlıklara taşınır (modülasyon + demodülasyon): aynı stil kontrolü, 'damla' (droplet) artefaktları olmadan.\")\n    R_.test(\"StyleGAN blokları\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        C, d = 10, 16\n        centers = rng.normal(0, 4, (C, d))\n\n        def probs(F):\n            D2 = ((F[:, None] - centers[None]) ** 2).sum(-1)\n            Z = -D2 / 2\n            Z -= Z.max(1, keepdims=True)\n            P = np.exp(Z)\n            return P / P.sum(1, keepdims=True)\n        real = centers[rng.integers(0, C, 5000)] + rng.normal(size=(5000, d))\n        cases = {\"gerçek dağılım\": real,\n                 \"sınıf başına tek 'mükemmel' örnek (merkezler)\": centers[rng.integers(0, C, 5000)],\n                 \"sınıf başına tek örnek, merkezden uzakta\": (centers + rng.normal(size=(C, d)))[rng.integers(0, C, 5000)],\n                 \"yarı modlar\": centers[rng.integers(0, C // 2, 5000)] + rng.normal(size=(5000, d)),\n                 \"sınıflar arası bulanık (merkezlerin karışımı)\": 0.5 * (centers[rng.integers(0, C, 5000)] + centers[rng.integers(0, C, 5000)]) + rng.normal(size=(5000, d))}\n        rows, r = [], {}\n        for name, F in cases.items():\n            r[name] = (need(ns, \"inception_score\")(probs(F)), need(ns, \"fid\")(real, F))\n            rows.append([name, f\"{r[name][0]:.2f}\", f\"{r[name][1]:.1f}\"])\n        ens = r[\"sınıf başına tek 'mükemmel' örnek (merkezler)\"]\n        ensure(ens[0] >= r[\"gerçek dağılım\"][0] - 0.05 and ens[1] > 5, \"IS kandırılır, FID kandırılmaz\")\n        ensure(r[\"yarı modlar\"][0] < 6 and r[\"sınıflar arası bulanık (merkezlerin karışımı)\"][0] < r[\"gerçek dağılım\"][0], \"IS çeşitlilik ve netlik kaybını görür\")\n        R_.table(\"10 sınıflı sentetik öznitelik uzayı (Inception vekili: merkezlere uzaklık softmax'ı)\", [\"üretilen\", \"IS (≤ 10)\", \"FID\"], rows,\n                 note=\"IS yalnız sınıflandırıcının 'emin mi' ve 'sınıflar dengeli mi' sorularına bakar: her sınıftan tek bir mükemmel görüntüyü sonsuza kadar tekrarlayan bir üretici en yüksek IS'i alır (sınıf içi çeşitlilik sıfır). FID gerçek verinin kovaryansıyla karşılaştırdığı için bunu cezalandırır (Barratt & Sharma 2018).\")\n    R_.test(\"IS vs FID\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        d = 32\n        A = rng.normal(size=(d, d)) / np.sqrt(d)\n        real = rng.normal(size=(50000, d)) @ A.T\n        good = lambda n, r: r.normal(size=(n, d)) @ A.T\n        bad = lambda n, r: 1.15 * r.normal(size=(n, d)) @ A.T + 0.05\n        Ns = np.array([500, 1000, 2000, 5000, 10000])\n        fg = need(ns, \"fid_curve\")(real, good, Ns, 1)\n        fb = ns[\"fid_curve\"](real, bad, Ns, 2)\n        ig, ib = need(ns, \"fid_inf\")(Ns, fg), ns[\"fid_inf\"](Ns, fb)\n        ensure(fg[0] > 2 * fg[-1] and abs(ig) < 0.05, f\"yanlılık ~1/N, FID∞ ≈ 0 ({ig:.3f})\")\n        S = A @ A.T\n        true_bad = np.sum(np.full(d, 0.05) ** 2) + np.trace(S) * (1 + 1.15 ** 2 - 2 * 1.15)\n        close(ib, true_bad, atol=0.05, what=\"FID∞ gerçek değere yakın\")\n        rows = [[int(n), f\"{a:.3f}\", f\"{b:.3f}\"] for n, a, b in zip(Ns, fg, fb)] + [[\"∞ (uydurma)\", f\"{ig:.3f}\", f\"{ib:.3f}\"]]\n        R_.table(\"32 boyutlu Gauss öznitelikler: örnek sayısı N'ye göre FID\", [\"N\", \"doğru üretici\", \"kusurlu üretici\"], rows,\n                 note=f\"Gerçek FID 0 olan üretici bile N = 500'de {fg[0]:.2f} alır; yanlılık ~1/N azalır ve modelden modele değişir — farklı N ile hesaplanmış FID'ler karşılaştırılamaz. Chong & Forsyth: FID'i birkaç N'de hesaplayıp 1/N'ye doğrusal uydurmak (FID∞) yansız bir tahmin verir (kusurlu üreticinin gerçek değeri {true_bad:.3f}).\")\n    R_.test(\"FID yanlılığı\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        old = rng.normal(size=(3, 4, 4))\n        new = rng.normal(size=(3, 8, 8))\n        f = need(ns, \"fade_in\")(old, new, 0.25)\n        close(f[:, 2, 3], 0.75 * old[:, 1, 1] + 0.25 * new[:, 2, 3], what=\"yumuşak geçiş\")\n        close(ns[\"fade_in\"](old, new, 1.0), new, what=\"α = 1 → yalnız yeni katman\")\n        X = rng.normal(size=(8, 4, 5, 5))\n        Y = need(ns, \"minibatch_std\")(X)\n        ensure(Y.shape == (8, 5, 5, 5) and np.allclose(Y[:, 4], Y[0, 4, 0, 0]), \"sabit ek kanal\")\n        close(Y[0, 4, 0, 0], X.std(0).mean(), atol=1e-6, what=\"batch std'lerinin ortalaması\")\n        Yc = ns[\"minibatch_std\"](np.repeat(X[:1], 8, 0))\n        ensure(Yc[0, 4, 0, 0] < 1e-3, \"çeşitliliği olmayan batch → ~0: ayırıcı mod çöküşünü görebilir\")\n        P = need(ns, \"pixel_norm\")(X * 10)\n        close(np.sqrt((P ** 2).mean(1)), np.ones((8, 5, 5)), atol=1e-6, what=\"piksel başına birim RMS\")\n        rows = []\n        for fan in (16, 256, 4096):\n            a, b = need(ns, \"equalized_step\")(fan, 1e-3, 100, 0)\n            rows.append([fan, f\"{a:.3f}\", f\"{b:.3f}\"])\n        ensure(float(rows[2][1]) > 5 * float(rows[0][1]) and abs(float(rows[2][2]) - float(rows[0][2])) < 0.01, \"eşitlenmiş lr: göreli adım fan-in'den bağımsız\")\n        R_.table(\"Adam benzeri (işaret) güncellemelerle 100 adım sonra etkin ağırlığın göreli değişimi\", [\"fan-in\", \"He başlatma\", \"N(0, 1) + çalışma anı çarpanı c\"], rows,\n                 note=\"Adam gradyanı kendi ölçeğine böler: adım boyu parametrenin büyüklüğünden bağımsızdır. He ile başlatılmış küçük ağırlıklar (büyük fan-in) bu yüzden göreli olarak çok daha hızlı değişir; katmanlar farklı hızlarda öğrenir. Eşitlenmiş öğrenme oranı (ProGAN) tüm ağırlıkları N(0, 1) tutar, ölçeği ileri geçişte uygular: her katmanın göreli öğrenme hızı aynı olur.\")\n    R_.test(\"ProGAN hileleri\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_7_gan_advanced as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Eğitim döngüsü:** her yinelemede önce ayırıcı (gerçek batch etiket 1 + üretilen batch etiket 0), sonra üretici (ayırıcı donuk, üretilenler etiket 1 ile — doymayan kayıp $-\log D(G(z))$; özgün $\log(1-D(G(z)))$ başlangıçta $D(G(z))\approx0$'da neredeyse sıfır gradyan verir). Yeterli kapasiteyle tek denge $p_g=p_{data}$, $D=\tfrac12$'dir — ama ulaşılacağının garantisi yoktur.

**Zorluklar:** sıfır toplamlı oyunda **Nash dengesi** (herkes soldan ya da herkes sağdan sürer — iki denge; avcı–av gibi karma stratejiler). **Mod çöküşü** (yukarıdaki quiz), parametre salınımları ve ani ıraksama, hiperparametre hassasiyeti. Önlemler: **deneyim tekrarı** (ayırıcıyı yalnız son üreticinin değil, bir tampondaki eski üretilenlerin de karışımıyla eğitmek), **mini-batch ayrımı** (batch içi benzerlik istatistiğini ayırıcıya vermek → çeşitliliği olmayan batch kolayca reddedilir).

**DCGAN** (Radford ve ark. 2015): havuz katmanları yerine adımlı konvolüsyon (ayırıcı) ve transposed konvolüsyon (üretici); üretici çıkışı ve ayırıcı girişi hariç BN; derin mimarilerde tam bağlı gizli katmanları kaldır; üreticide ReLU (çıkış tanh), ayırıcıda LeakyReLU. Kurallar her zaman işlemez; bazen aynı modeli başka tohumla eğitmek yeter.

**Ne zaman durmalı / değerlendirme:** GAN örtük bir dağılımdır, olabilirlik vermez. **Inception Score:** önceden eğitilmiş Inception v3 ile — görüntü kalitesi (tek bir örnek için p(y|x) düşük entropili) ve çeşitlilik (tüm örnekler üzerinde p(y) yüksek entropili): $\mathrm{IS}=\exp\mathbb E_x\mathrm{KL}(p(y|x)\Vert p(y))$. **FID:** son havuz katmanı öznitelikleri; gerçek ve üretilen öznitelikler çok değişkenli Gauss varsayılır ve Fréchet (Wasserstein-2) uzaklığı hesaplanır; düşük FID = iyi.

**ProGAN** (Karras ve ark. 2018): 4 × 4'ten 1024 × 1024'e katman ekleyerek büyüyen üretici ve ayırıcı (yığılmış AE'lerin katman katman eğitimine benzer; eski katmanlar eğitilebilir kalır); yeni katmanlar yumuşak geçişle girer. Ek olarak **mini-batch standart sapma katmanı**, **eşitlenmiş öğrenme oranı** ve **piksel normalizasyonu**. **StyleGAN** (2019): sabit öğrenilen 4 × 4 girdi; z'yi 8 katmanlı MLP ile w'ye eşleyen **eşleme ağı**; her ölçekte w'den türetilen stil ($y_s,y_b$) ile **AdaIN** $=y_s\frac{x-\mu(x)}{\sigma(x)}+y_b$; her katmana ölçeklenmiş piksel gürültüsü (saç, gözenek gibi stokastik ayrıntı); **stil karıştırma** (iki w'yi farklı katmanlarda kullanmak) ve **kırpma** ($\bar w+\psi(w-\bar w)$: kalite ↑, çeşitlilik ↓).

## 1 · Elle

<!-- çözüm -->

FID 1B: $(\mu_r-\mu_g)^2+\sigma_r^2+\sigma_g^2-2\sigma_r\sigma_g=1+1+4-4=2$ (= $(\Delta\mu)^2+(\sigma_r-\sigma_g)^2$). 2B köşegen: $1+(1+4-2\sqrt4)+(1+9-2\sqrt9)=1+1+4=6$; köşegen kovaryanslarda iz terimi $\sum_i(\sigma_{r,i}-\sigma_{g,i})^2$'dir.

IS: marjinal (0.5, 0.5); her örnek için KL = $0.9\ln1.8+0.1\ln0.2=0.368$ → IS = $e^{0.368}=1.445$ (en fazla sınıf sayısı = 2; tamamen emin ve dengeli üretici 2 alırdı).

AdaIN: μ = 2.5, σ = √1.25 = 1.118; (4 − 2.5)/1.118 = 1.342 → 2 · 1.342 + 1 = 3.683. Kanal artık ortalama 1, std 2 taşır: önceki stil bilgisi silinir, yenisi yazılır.

Kırpma: 0.7 · 2 = 1.4. Yumuşak geçiş: 0.7 · 2 + 0.3 · 4 = 2.6 (α eğitim boyunca 0 → 1). Üretici: (4 − 1) · 2 − 2 + 4 = 8.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**FID ve IS**

- `fid1`: 1B: gerçek N(0, 1), üretilen N(1, 2²) → Fréchet uzaklığı
- `fid2`: 2B: $\mu_r-\mu_g=(1,0)$, $\Sigma_r=I$, $\Sigma_g=\mathrm{diag}(4,9)$
- `iscore`: IS: örneklerin yarısında p(y|x) = (0.9, 0.1), yarısında (0.1, 0.9)

**StyleGAN ve ProGAN**

- `adain`: AdaIN: kanal değerleri 1, 2, 3, 4 (popülasyon std), stil $y_s=2$, $y_b=1$ → 4'ün çıktısı
- `trunc`: Kırpma: $\bar w=0$, ψ = 0.7, w = 2 → $\bar w+\psi(w-\bar w)$
- `fade`: Yumuşak geçiş α = 0.3: büyütülmüş eski çıktı 2, yeni katman 4 → $(1-\alpha)\,\text{eski}+\alpha\,\text{yeni}$
- `up`: DCGAN üretici: 4 × 4 → ConvTranspose(k = 4, s = 2, p = 1) → kenar

In [ ]:
cevaplar = dict(
    fid1=None,
    fid2=None,
    iscore=None,
    adain=None,
    trunc=None,
    fade=None,
    up=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"fid1":[2,0],"fid2":[6,0],"iscore":[1.4449348111684153,0.000001],"adain":[3.6832815729997477,0.000001],"trunc":[1.4,1e-9],"fade":[2.6,1e-9],"up":[8,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `frechet_1d(m1, s1, m2, s2)` — standart sapmalarla.
- `inception_score(P)` — `P`: her örnek için p(y|x) listesi.
- `adain(x, ys, yb, eps=0.0)` — tek kanal, popülasyon std.
- `truncate(w, w_avg, psi)`, `fade_in(old, new, alpha)` (listeler).
- `he_scale(fan_in, gain=2 ** 0.5)` — eşitlenmiş öğrenme oranı çarpanı $\text{gain}/\sqrt{\text{fan\_in}}$.

In [ ]:
%%writefile stage_b.py
import math


def frechet_1d(m1, s1, m2, s2):
    ...


def inception_score(P):
    ...


def adain(x, ys, yb, eps=0.0):
    ...


def truncate(w, w_avg, psi):
    ...


def fade_in(old, new, alpha):
    ...


def he_scale(fan_in, gain=2 ** 0.5):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy.

- `sqrtm_psd(A)` — simetrik yarı pozitif tanımlı matrisin karekökü (`eigh`, negatif özdeğerleri 0'a kırp). Döngüsüz.
- `sqrtm_ns(A, iters=50)` — Newton–Schulz yinelemesi: $Y_0=A/\|A\|_F$, $Z_0=I$, $T=\tfrac12(3I-Z_kY_k)$, $Y_{k+1}=Y_kT$, $Z_{k+1}=TZ_k$; sonuç $Y\sqrt{\|A\|_F}$ (tek döngü).
- `frechet(mu1, S1, mu2, S2)` — iz terimini simetrik biçimle: $\mathrm{tr}\sqrt{S_1^{1/2}S_2S_1^{1/2}}$.
- `fid(F1, F2)` — öznitelik matrislerinden (satır = örnek; kovaryans `ddof=1`).
- `inception_score(P, splits=1)` — `P (N, C)`; batch'leri `splits` parçaya bölüp skorların ortalaması ve std'si.

In [ ]:
%%writefile stage_c.py
import numpy as np


def sqrtm_psd(A):
    ...


def sqrtm_ns(A, iters=50):
    ...


def frechet(mu1, S1, mu2, S2):
    ...


def fid(F1, F2):
    ...


def inception_score(P, splits=1):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Veri: 16 × 16'ya büyütülmüş rakamlar, [−1, 1]. Değerlendirme için verilen küçük bir CNN sınıflandırıcının sondan bir önceki katmanı "Inception" vekili.

- `Generator(z=32, c=64)` — DCGAN: ConvT(z → 2c, 4, 1, 0) → BN → ReLU → ConvT(2c → c, 4, 2, 1) → BN → ReLU → ConvT(c → 1, 4, 2, 1) → tanh. Girdi `(B, z)`.
- `Discriminator(c=64)` — Conv(1 → c, 4, 2, 1) → LeakyReLU(0.2) → Conv(c → 2c, 4, 2, 1) → BN → LeakyReLU → Conv(2c → 1, 4, 1, 0); BN girişte yok. Çıkış logit `(B,)`.
- `train_gan(G, D, X, epochs, z=32, lr=2e-4)` — Adam(β₁ = 0.5); önce ayırıcı adımı, sonra doymayan üretici adımı.
- `fid_np(F1, F2)` — c aşamasındaki FID (torch tensörlerinden numpy'a çevirerek).

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits


def digits16():
    X, y = load_digits(return_X_y=True)
    X = torch.tensor(X, dtype=torch.float32).view(-1, 1, 8, 8) / 16
    return F.interpolate(X, size=16, mode="bilinear", align_corners=False) * 2 - 1, torch.tensor(y)


class Generator(nn.Module):
    def __init__(self, z=32, c=64):
        super().__init__()
        ...

    def forward(self, z):
        ...


class Discriminator(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        ...

    def forward(self, x):
        ...


def train_gan(G, D, X, epochs, z=32, lr=2e-4):
    ...


def fid_np(F1, F2):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

StyleGAN yapı taşları.

- `MappingNet(z=64, w=64, L=8)` — piksel normu ($z/\sqrt{\mathrm{mean}(z^2)+\epsilon}$) + L × (doğrusal + LeakyReLU 0.2).
- `AdaIN(C, w)` — w'den doğrusal izdüşümle `(y_s, y_b)` (`y_s` başlangıçta 1 civarı: bias 1), örnek ve kanal başına normalizasyon.
- `NoiseInjection(C)` — kanal başına öğrenilen ölçekle `x + s_c · noise`.
- `modulated_conv(x, weight, style, demod=True)` — StyleGAN2: $w'_{ijk}=s_i\,w_{ijk}$, $w''=w'/\sqrt{\sum_{i,k}w'^2+\epsilon}$; örnek başına ağırlık → grup konvolüsyonu (batch'i gruplara katlayarak).
- `truncate(w, w_avg, psi)`.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class MappingNet(nn.Module):
    def __init__(self, z=64, w=64, L=8):
        super().__init__()
        ...

    def forward(self, z):
        ...


class AdaIN(nn.Module):
    def __init__(self, C, w):
        super().__init__()
        ...

    def forward(self, x, w):
        ...


class NoiseInjection(nn.Module):
    def __init__(self, C):
        super().__init__()
        ...

    def forward(self, x, noise=None):
        ...


def modulated_conv(x, weight, style, demod=True):
    ...


def truncate(w, w_avg, psi):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · IS'i kandırmak, FID'i kandıramamak

NumPy. Sentetik "öznitelik uzayı": 10 sınıf, her sınıf bir Gauss bulutu; sınıflandırıcı = sınıf merkezlerine uzaklık softmax'ı (verilir).

- `inception_score(P)` (tek parça) ve `fid(F1, F2)` — c'dekilerin kopyası.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def inception_score(P):
    ...


def fid(F1, F2):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · FID'in örneklem yanlılığı ve FID∞

NumPy.

- `fid(F1, F2)` (c'deki gibi).
- `fid_curve(F_real, sampler, Ns, seed)` — her N için üreticiden N örnek çekip gerçek kümeyle FID.
- `fid_inf(Ns, fids)` — FID'i 1/N'ye karşı doğrusal uydur, kesişim (N → ∞) değerini döndür.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def fid(F1, F2):
    ...


def fid_curve(F_real, sampler, Ns, seed):
    ...


def fid_inf(Ns, fids):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · ProGAN hileleri: yumuşak geçiş, mini-batch std, piksel normu, eşitlenmiş öğrenme oranı

NumPy.

- `fade_in(old_lowres, new_highres, alpha)` — eski çıktıyı en yakın komşu 2× büyüt, karıştır. `(C, H, W)`.
- `minibatch_std(X, eps=1e-8)` — `X (N, C, H, W)`: her öznitelik/konum için batch std'si, hepsinin ortalaması → tek değer; her örneğe sabit bir kanal olarak ekle → `(N, C + 1, H, W)`.
- `pixel_norm(X, eps=1e-8)` — her piksel konumunda kanal vektörünü birim RMS'e normalize et.
- `equalized_step(fan_in, lr, steps, seed)` — Adam benzeri ölçekten bağımsız güncelleme altında: (a) ağırlık doğrudan He ölçeğinde başlatılmış, (b) N(0, 1) başlatılıp çalışma anında $c=\sqrt{2/\text{fan\_in}}$ ile çarpılmış; her durumda etkin ağırlığın göreli değişimi `(a, b)`.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def fade_in(old_lowres, new_highres, alpha):
    ...


def minibatch_std(X, eps=1e-8):
    ...


def pixel_norm(X, eps=1e-8):
    ...


def equalized_step(fan_in, lr, steps, seed):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Radford ve ark. (2016): DCGAN ve gizli uzay aritmetiği

[Radford, Metz, Chintala — Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks (ICLR 2016)](https://arxiv.org/abs/1511.06434)

d aşamasındaki DCGAN'ı CelebA'nın 64 × 64 bir alt kümesinde (ya da Fashion-MNIST'te) eğit; makalenin Şekil 7'sindeki vektör aritmetiğini (gülümseyen kadın − nötr kadın + nötr erkek) her kavram için 3 örneğin z ortalamasıyla yap. Ayırıcının öznitelikleri üzerinde doğrusal SVM ile denetimsiz temsil kalitesini ölç (Tablo 1). Eğitim boyunca FID'i çiz ve kuralların ablasyonunu yap: BN'siz, havuzlu, ReLU'lu ayırıcı.


**Adımlar**

1. FID için c aşamasındaki fid'i, öznitelik çıkarıcı olarak önceden eğitilmiş bir sınıflandırıcıyı kullan.

**Kabul ölçütleri**

- Vektör aritmetiği anlamlı görüntüler üretir.
- Kurallardan sapan varyantlar daha kararsız ve daha yüksek FID'li.

## Makale ödevi: Karras ve ark. (2019): mini StyleGAN

[Karras, Laine, Aila — A Style-Based Generator Architecture for Generative Adversarial Networks (CVPR 2019)](https://arxiv.org/abs/1812.04948)

e aşamasındaki blokları birleştirerek 32 × 32 bir stil tabanlı üretici kur (sabit 4 × 4 girdi, eşleme ağı, AdaIN, gürültü), ProGAN ayırıcısıyla eğit. Makaledeki analizleri tekrarla: stil karıştırma (kaba/orta/ince katmanlarda kaynak B'den stil, Şekil 3), gürültünün etkisi (Şekil 4–5), kırpma ψ'ye göre FID–çeşitlilik ödünleşimi ve algısal yol uzunluğu (PPL, Bölüm 4.1) ile z ve w uzaylarının karşılaştırması.


**Adımlar**

1. PPL: ε = 10⁻⁴ adımlı interpolasyonda ardışık görüntüler arasındaki öznitelik uzaklığı.

**Kabul ölçütleri**

- Kaba katmanlarda karıştırma pozu/şekli, ince katmanlarda rengi/dokuyu değiştirir.
- w uzayında PPL, z uzayındakinden düşük.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 17](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Radford, Metz, Chintala — Unsupervised Representation Learning with DCGANs (ICLR 2016)](https://arxiv.org/abs/1511.06434) — arXiv
- [Salimans ve ark. — Improved Techniques for Training GANs (NeurIPS 2016)](https://arxiv.org/abs/1606.03498) — arXiv
- [Heusel ve ark. — GANs Trained by a Two Time-Scale Update Rule (FID, NeurIPS 2017)](https://arxiv.org/abs/1706.08500) — arXiv
- [Karras ve ark. — Progressive Growing of GANs (ICLR 2018)](https://arxiv.org/abs/1710.10196) — arXiv
- [Karras, Laine, Aila — A Style-Based Generator Architecture for GANs (CVPR 2019)](https://arxiv.org/abs/1812.04948) — arXiv
- [Karras ve ark. — Analyzing and Improving the Image Quality of StyleGAN (CVPR 2020)](https://arxiv.org/abs/1912.04958) — arXiv
- [Chong & Forsyth — Effectively Unbiased FID and Inception Score (CVPR 2020)](https://arxiv.org/abs/1911.07023) — arXiv
- [Stanford CS236 — Deep Generative Models: GAN değerlendirmesi](https://deepgenerativemodels.github.io/) — ©